# Submission: mlp_4out_emb_bn before tuning, with dropout and 20 epochs

Report: [mlp_report.md](../../mlp_report.md#final-comparison)


In [ ]:
import os
import sys

import numpy as np
import torch

sys.path.insert(0, "../../..")  # the ashrae package lives in the homework folder
from ashrae.config import PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features
from ashrae.modeling import MeterMLP, make_loaders
from ashrae.submission import build_submission, check_submission, to_readings
from ashrae.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
CHUNK_SIZE = 1_000_000  # test rows predicted at once
EMB_DIM = 16  # exp02, before exp03 picked 12
DROPOUT = 0.1  # exp15
if SMOKE:
    EXPERIMENT = "ashrae-submission-smoke"
    N_TRAIN = 20_000
    N_TEST = 50_000
    EPOCHS = 1
    SUBMISSION_PATH = None  # partial test set, so nothing to submit
else:
    EXPERIMENT = "ashrae-submission"
    N_TRAIN = 2_000_000  # the same row count as the other submissions, spread over all of 2016
    N_TEST = None  # all 41.7M test rows
    EPOCHS = 20  # exp02 trained 10 and was still improving
    SUBMISSION_PATH = PROCESSED / "submission_mlp_4out_emb_bn_dropout.csv"

experiment = Experiment(EXPERIMENT, "submission_mlp_bn_dropout")

## Training rows


In [ ]:
df = read_processed()
n_buildings = int(df.building_id.max()) + 1
train_df = df[~df.anomaly].sample(
    N_TRAIN, random_state=0
)  # all of 2016, anomalies dropped
del df

year_median = train_df.year_built.median()  # reused for the test set
X_train = build_features(train_df, year_median)
train_loader, _, scaler = make_loaders(train_df, None, X_train, None)
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features")

## Model


In [ ]:
# Adam at lr 1e-3: the default of EnergyModel
model, run_id = experiment.fit(
    lambda: MeterMLP(X_train.shape[1], n_buildings, emb_dim=EMB_DIM, batch_norm=True, dropout=DROPOUT),
    train_loader,
    run_name="mlp_4out_emb_bn | dropout, 20 epochs | all 2016",
    epochs=EPOCHS,
    params={"n_train": N_TRAIN, "emb_dim": EMB_DIM, "dropout": DROPOUT},
    tags={"model": "mlp_4out_emb_bn"},
)
model = model.eval()
print(
    f"mlp_4out_emb_bn, dropout: training RMSLE {experiment.history(run_id, 'train_rmsle')[-1]:.3f}"
)

## Test predictions


In [ ]:
test_df = read_processed("test_merged.parquet")
if N_TEST:
    test_df = test_df.head(N_TEST)


@torch.no_grad()
def predict_log(rows):
    """Clipped log1p predictions for `rows`, built with the same feature columns and scaler as training."""
    x = scaler.transform(
        build_features(rows, year_median).reindex(columns=X_train.columns, fill_value=0)
    )
    batch = {
        "x": torch.tensor(x, dtype=torch.float32, device=model.device),
        "meter": torch.tensor(
            rows.meter.to_numpy(), dtype=torch.long, device=model.device
        ),
        "building": torch.tensor(
            rows.building_id.to_numpy(), dtype=torch.long, device=model.device
        ),
    }
    return model(batch).clamp(min=0).cpu().numpy()


starts = range(0, len(test_df), CHUNK_SIZE)
mlp_log = np.concatenate(
    [predict_log(test_df.iloc[start : start + CHUNK_SIZE]) for start in starts]
)
print(f"mean log1p prediction: {mlp_log.mean():.3f}")

In [ ]:
reading = to_readings(mlp_log, test_df)  # site 0 electricity back in kBTU
submission = build_submission(test_df, reading)
if SUBMISSION_PATH:
    check_submission(submission)
    submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.4f")
    print(f"saved {SUBMISSION_PATH}: {len(submission):,} rows")
submission.meter_reading.describe().round(2).to_frame()

In [ ]:
if SUBMISSION_PATH:  # multithreaded 7-Zip CLI (brew install sevenzip), much faster than py7zr
    !cd "{SUBMISSION_PATH.parent}" && rm -f "{SUBMISSION_PATH.stem}.7z" && 7zz a -mmt=on "{SUBMISSION_PATH.stem}.7z" "{SUBMISSION_PATH.name}"

## Settings

- One output per meter type
- Building identity as a learned embedding of size 16
- 2 hidden layers of 64, ReLU, BatchNorm, dropout 0.1
- Adam at lr 1e-3, batch 4096, 20 epochs
